# Phase 5 — Frequency-Aware Transformer Encoder (FATE)

An interactive educational walkthrough of **Phase 5: Frequency-Aware Transformer Encoder (FATE)** for Motor Imagery EEG Classification.
This notebook demonstrates Band x Channel tokenization, 2D hierarchical positional embeddings (Band + Channel), TransformerEncoder sequence processing, token mapping tables, and contextual token embedding visualizations.

## 1. Objective

Phase 5 introduces the first sequence model of our architecture to capture global, non-local dependencies across multi-band spatial EEG representations.

- **Core Principle**: Tokenization preserves the EEG frequency-band hierarchy. Each token represents **one EEG channel within one frequency band**.
- **What FATE Solves**: While Adaptive Channel Attention (ACA) refines channel features locally within each frequency band, FATE models global self-attention interactions between all tokens across all sub-bands (Theta, Alpha, Beta, Gamma) and electrodes.
- **Interface Contract for Phase 6**: FATE outputs contextual token embeddings of shape (B, N, d_model) where N = F x C (532 tokens for HGD). It contains no pooling or classifier heads, establishing a clean interface for Phase 6.

## 2. Theory & Mathematical Formulation

### Tokenization Strategy:
Given refined ACA tensor $X \in \mathbb{R}^{B \times F \times C \times S}$, we define token $k$ at frequency band $f \in \{0 \dots F-1\}$ and channel $c \in \{0 \dots C-1\}$ using deterministic ordering:

$$k = f \cdot C + c$$
$$N = F \times C$$

Reshaping $X$ yields token sequence tensor $T \in \mathbb{R}^{B \times N \times S}$.

### 2D Hierarchical Positional Embeddings:
Standard 1D sinusoidal positional encodings ignore dual spatial-spectral identity. Every token embedding combines sample projection with learnable Band and Channel embeddings:

$$E_k = \text{TemporalProjection}(S_k) + E_{\text{band}}[f] + E_{\text{chan}}[c] \in \mathbb{R}^{d_{\text{model}}}$$

Where $E_{\text{band}} \in \mathbb{R}^{F \times d_{\text{model}}}$ and $E_{\text{chan}} \in \mathbb{R}^{C \times d_{\text{model}}}$.

### Multi-Head Self-Attention:
Embedded token sequence $E \in \mathbb{R}^{B \times N \times d_{\text{model}}}$ passes through PyTorch `TransformerEncoder` layers:

$$Z^{(0)} = E$$
$$Z^{(l)} = \text{TransformerEncoderLayer}(Z^{(l-1)}) \quad \text{for } l = 1 \dots L$$
$$\text{ContextualEmbeddings} = Z^{(L)} \in \mathbb{R}^{B \times N \times d_{\text{model}}}$$

## 3. Tokenization Transformation

```text
Input ACA Tensor: (B, F, C, S) -> (B, 4, 133, 250)
                       │
                       ▼  [BandChannelTokenizer: k = f*C + c]
Token Sequence:   (B, N, S) -> (B, 532, 250)
                       │
                       ▼  [TemporalProjection: Linear(250 -> 128)]
Projected Tokens: (B, N, d_model) -> (B, 532, 128)
                       │
                       ▼  [+ BandEmbedding(F, 128) + ChannelEmbedding(C, 128)]
2D Position Embed:(B, N, d_model) -> (B, 532, 128)
                       │
                       ▼  [TransformerEncoder Stack (4 layers, 8 heads)]
Contextual Embed: (B, N, d_model) -> (B, 532, 128)
```

## 4. Architecture

Decoupled module hierarchy:
1. `BandChannelTokenizer` (`models/transformer/tokenizer.py`)
2. `TemporalProjection` (`models/common/temporal_projection.py`)
3. `BandChannelEmbedding` (`models/transformer/embeddings.py`)
4. `FrequencyAwareTransformer` / `FATE` (`models/transformer/frequency_aware_transformer.py`)

## 5. Implementation

Let's import FATE modules, load configuration from `configs/model.yaml`, and instantiate the pipeline.

In [ ]:
import os
import sys
import torch
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

def get_project_root():
    curr = os.path.abspath(os.getcwd())
    while curr and not os.path.exists(os.path.join(curr, "models")):
        parent = os.path.dirname(curr)
        if parent == curr:
            break
        curr = parent
    return curr

PROJECT_ROOT = get_project_root()
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f"[OK] Project Root set to: {PROJECT_ROOT}")

from models.attention import ACA, AdaptiveChannelAttentionConfig
from models.transformer import (
    FATE,
    FrequencyAwareTransformer,
    FrequencyAwareTransformerConfig,
    BandChannelTokenizer,
    BandChannelEmbedding,
    FATEOutput,
)

print("[OK] Successfully imported FATE, ACA, and Transformer modules.")

## 6. Verification Demo (ACA -> FATE Pipeline)

Passing a 4D EEG tensor `(Batch=2, Bands=4, Channels=133, Samples=250)` through ACA and FATE.

In [ ]:
# Load configurations
model_yaml = os.path.join(PROJECT_ROOT, "configs", "model.yaml")
aca_cfg = AdaptiveChannelAttentionConfig.from_yaml(model_yaml)
fate_cfg = FrequencyAwareTransformerConfig.from_yaml(model_yaml)

aca_module = ACA(config=aca_cfg, num_channels=133, num_bands=4)
fate_module = FATE(config=fate_cfg)

aca_module.eval()
fate_module.eval()

torch.manual_seed(42)
x_raw = torch.randn(2, 4, 133, 250)

# Pipeline execution
x_aca = aca_module(x_raw)  # (2, 4, 133, 250)
contextual_embeddings, fate_out = fate_module(x_aca, return_metadata=True, band_names=["Theta", "Alpha", "Beta", "Gamma"])

print("\n--- FATE Execution Summary ---")
print(f"Raw EEG Input Shape:        {x_raw.shape}")
print(f"ACA Refined Feature Shape:  {x_aca.shape}")
print(f"Contextual Embedding Shape: {contextual_embeddings.shape}")
print(f"Total Tokens Processed N:   {fate_out.token_metadata.num_tokens}")
print(f"Embedding Dimension d_model:{contextual_embeddings.size(-1)}")

## 7. Visualization & Token Mapping Inspection

### Complete Token Mapping Table
Displaying the deterministic mapping $k = f \cdot C + c$ between token index $k$ and its originating frequency band and channel.

In [ ]:
meta = fate_out.token_metadata
mapping_data = []
for m in meta.mappings:
    mapping_data.append({
        "Token Index": m.token_index,
        "Band Index": m.band_index,
        "Channel Index": m.channel_index,
        "Band Name": m.band_name,
        "Channel Name": m.channel_name,
    })

df_tokens = pd.DataFrame(mapping_data)
print(f"Total Mapped Tokens: {len(df_tokens)}")
print("\nSample Token Index Mappings (Beginning, Band Transitions, End):")
sample_rows = pd.concat([df_tokens.head(5), df_tokens.iloc[131:136], df_tokens.tail(5)])
display(sample_rows)

### Learned Band & Channel Positional Embeddings
Visualizing learned Band Embeddings $E_{\text{band}} \in \mathbb{R}^{4 \times 128}$ and Channel Embeddings $E_{\text{chan}} \in \mathbb{R}^{133 \times 128}$.

In [ ]:
b_weights = fate_module.embedding.band_embedding.weight[:4, :].detach().numpy()  # (4, 128)
c_weights = fate_module.embedding.channel_embedding.weight[:133, :].detach().numpy() # (133, 128)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8))

im1 = ax1.imshow(b_weights, aspect='auto', cmap='plasma')
ax1.set_yticks(range(4))
ax1.set_yticklabels(["Theta (4-8 Hz)", "Alpha (8-13 Hz)", "Beta (13-30 Hz)", "Gamma (30-38 Hz)"])
ax1.set_xlabel("Embedding Dimension d_model (0 to 127)")
ax1.set_title("Learned Frequency Band Positional Embeddings (4 x 128)")
plt.colorbar(im1, ax=ax1)

im2 = ax2.imshow(c_weights.T, aspect='auto', cmap='magma')
ax2.set_xlabel("EEG Channel Index (0 to 132)")
ax2.set_ylabel("Embedding Dimension d_model")
ax2.set_title("Learned EEG Channel Positional Embeddings (128 x 133)")
plt.colorbar(im2, ax=ax2)

plt.tight_layout()
plt.show()

### Contextual Token Embedding Inspection
Visualizing the output contextual EEG token embeddings produced by FATE across 532 tokens for batch item 0.

In [ ]:
context_np = contextual_embeddings[0].detach().numpy()  # (532, 128)

plt.figure(figsize=(14, 5))
plt.imshow(context_np, aspect='auto', cmap='viridis')
plt.colorbar(label="Embedding Activation")
plt.axhline(133, color='white', linestyle='--', linewidth=1.2, label="Band Transitions (133, 266, 399)")
plt.axhline(266, color='white', linestyle='--', linewidth=1.2)
plt.axhline(399, color='white', linestyle='--', linewidth=1.2)
plt.xlabel("Embedding Dimension d_model (0 to 127)")
plt.ylabel("Token Index k (0 to 531: Theta, Alpha, Beta, Gamma)")
plt.title("FATE Contextualized EEG Token Embeddings (532 x 128)")
plt.legend(loc="upper right")
plt.tight_layout()
plt.show()

## 8. Conclusion & Phase 5 Status

### Key Takeaways:
1. **Band x Channel Tokenization**: Successfully tokenized 4D multi-band EEG tensors `(B, 4, 133, 250)` into 532 structured tokens `(B, 532, 250)` with deterministic mapping $k = f \cdot C + c$.
2. **2D Hierarchical Embeddings**: Combined temporal sample projection with learnable Band and Channel positional embeddings ($E_k = \text{Proj}(S_k) + E_{\text{band}}[f] + E_{\text{chan}}[c]$).
3. **Contextual Self-Attention**: TransformerEncoder stack models global non-local interactions across spatial electrodes and spectral sub-bands, producing contextual embeddings `(B, 532, 128)`.
4. **Decoupled Architecture**: Tokenizer, TemporalProjection, Embeddings, and Transformer Encoder operate as single-responsibility modules.

**Phase 5 is complete and fully validated.** The architecture is now ready for **Phase 6 (Classification & Training Pipeline)**.